<center><h1 style="margin-bottom: 0px;">Artificial Intelligence (25/26)</h1></center>
<center><h2 style="margin-top: 0px;">Decision Trees Implementation </h2></center>

#### <center> Joana Antunes (202405702), Sílvia Pinto (202405988) </center> <br>

#### **5. Decision Trees** <br>
<div style="text-align: justify;"><p style="text-indent: 2em;">Decision trees are one of the simplest models and most interpretable machine learning algorithms. The type of decision trees we are going to implement is ID3(Iterative Dichotomiser 3), which builds the tree recursively by always choosing the attribute that provides the most information about the correct classification of the object. We start by implementing some functions that we will need. The first one is the entropy function, which represents the level of uncertainty over a certain variable. The entropy is obtained by the formula:</div>

$$H(X) = \sum_{i=1}^{n} -P(x_i) \log_2 P(x_i)$$
In this formula, $P(x_i)$ is the proportion of examples belonging to class $x_i$. When all examples belong to the same class, the entropy is 0 and when the example are equally split between the classes the entropy is 1.


In [ ]:
from collections import Counter
import math

def entropy(labels):
    n = len(labels)
    result = 0.0
    if n == 0: return result
    counts = Counter(labels)  # counts how many times each class appears
    for count in counts.values():
        p = count / n           # probability of this class
        result += -p * math.log2(p)
    return result

<div style="text-align: justify;"><p style="text-indent: 2em;">Next, we define the information gain function - it is the reduction in entropy achieved by splitting the examples according to a certain attribute:</div>

$$H(X) = \sum_{i=1}^{n} -P(x_i) \log_2 P(x_i)$$

In [ ]:
def information_gain(data, labels, attribute):
    n = len(labels)
    parent_entropy = entropy(labels)  # H(C)=entropy before the split
    groups = {}                       # Group examples by the value of the attribute
    for i, row in enumerate(data):
        val = row[attribute]
        if val not in groups:
            groups[val] = []
        groups[val].append(labels[i])
    weighted_entropy = 0.0            # H(C|A) — weighted average entropy after the split
    for group_labels in groups.values():
        weight = len(group_labels) / n
        weighted_entropy += weight * entropy(group_labels)
    return parent_entropy - weighted_entropy

<div style="text-align: justify;"><p style="text-indent: 2em;">Now, we define the the ID3 algorithm itself. The ID3 aproach always chooses the attribute with the highest information gain as the next splitting node. This process is applied recursively until one of the next base cases occurs:</div>

> &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**1.** *All examples have the same class;*

> &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**2.** *There are no more attributes to split on;*

> &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**3.** *There are no more examples left in a branch;*

> &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**4.** *It reached a maxium depht.*

<div style="text-align: justify;"><p style="text-indent: 2em;">The tree implemented is built using two components. The first is the DecisionNode class, which represents a node in the decision tree. A node can only be an internal node, which means it has an attribute to split on and a set of children, or a leaf node, which means it has a class label and represents a final classification. The second is the ID3 function, which builds the tree recursively. At each iteration this function selects the attribute with the highest information gain and creates a branch for each of its possible values. The algorithym is apllied again to each subset of examples and the recursion only stops when the bases cases refered previously are reached. In addition, the max_depth parameter is important for the Pop Out dataset, because its complexity is extremely high and that could lead to the tree learning general patterns instead of making predictions.</div>


In [ ]:
class DecisionNode:
    def __init__(self, attribute=None, label=None):
        self.attribute = attribute  # attribute to split on (internal node)
        self.label = label          # class label (leaf node)
        self.children = {}          # value = child node

    def is_leaf(self):
        return self.label is not None

def id3(data, labels, attributes, max_depth=None, depth=0):
    if len(set(labels)) == 1: return DecisionNode(label=labels[0]) # Base case 1: all examples have the same class
    if not attributes or (max_depth and depth >= max_depth): # Base case 2 and 4: no attributes left or max depth reached
        most_common = Counter(labels).most_common(1)[0][0]
        return DecisionNode(label=most_common)
    # Choose the attribute with the highest information gain
    best_attr = max(attributes, key=lambda a: information_gain(data, labels, a))
    node = DecisionNode(attribute=best_attr)
    remaining_attrs = attributes - {best_attr}
    values = set(row[best_attr] for row in data) # Get all possible values of the best attribute
    for val in values:
        # Split examples by this value
        subset_data   = [data[i]   for i in range(len(data)) if data[i][best_attr] == val]
        subset_labels = [labels[i] for i in range(len(data)) if data[i][best_attr] == val]
        if not subset_data: # Base case 3: no examples for this branch
            most_common = Counter(labels).most_common(1)[0][0] # use most common label of the parent
            node.children[val] = DecisionNode(label=most_common)
        else: node.children[val] = id3(subset_data, subset_labels, remaining_attrs, max_depth, depth + 1) # Recursive
    return node

<div style="text-align: justify;"><p style="text-indent: 2em;">After building the tree, we also need a function to classify new unseen examples. The following "classify" function is responsable for that. It traverses the tree from the root and at each iternal node reads the value of the corresponding attribute from the sample, following the matching branch. It returns a prediction when it reaches a leaf node.</div>

In [ ]:
def classify(node, sample, default_label=None):
    if node.is_leaf(): return node.label
    val = sample.get(node.attribute)
    if val not in node.children: return default_label
    return classify(node.children[val], sample, default_label)